# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Print summary from metadata
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s.

In [ ]:
# List all record sets (`@id`s and names), then list their fields and columns with IDs
if hasattr(dataset.metadata, 'record_sets'):
    record_sets = dataset.metadata.record_sets
elif hasattr(dataset.metadata, 'recordSet'):
    # Some Croissant schemas use PascalCase
    record_sets = dataset.metadata.recordSet
else:
    record_sets = []

print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for rs in record_sets:
    # Each rs is a mlcroissant.datastructures.metadata.RecordSet
    print(f"- Recordset name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    record_set_ids.append(rs.id)

    # Show fields
    if hasattr(rs, 'fields') and rs.fields:
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
        print()
    # Show columns
    if hasattr(rs, 'columns') and rs.columns:
        print(f"  Columns:")
        for col in rs.columns:
            print(f"    - {col.name} (@id: {col.id}, type: {col.data_type})")
        print()
if not record_set_ids:
    print("No record sets found or the record sets are not defined in the Croissant schema.")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Set up record set(s) for extraction
import warnings
record_sets_to_load = record_set_ids  # Use all found recordset IDs
dataframes = dict()

for rs_id in record_sets_to_load:
    try:
        # Records is an iterator of dictionaries
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from RecordSet '@id': {rs_id}")
        else:
            print(f"No records found for RecordSet '@id': {rs_id}")
    except Exception as e:
        print(f"Failed to load data for record set {rs_id}: {e}")

# For demonstration, pick the first non-empty dataframe
main_rs_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_rs_id = k
        break

if main_rs_id:
    print(f"\nColumns in main record set (@id: {main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes available. Check if the dataset exposes record sets and data.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. This section demonstrates EDA steps using actual field `@id`s from the selected record set.

In [ ]:
# Determine a numeric field (by @id) for filtering and normalization
# Below we attempt to auto-select a likely numeric field
import numpy as np

df = dataframes.get(main_rs_id)

numeric_field_id = None
candidate_cols = []
if df is not None:
    for col in df.columns:
        # Only consider columns that can be numeric after conversion
        try:
            numeric = pd.to_numeric(df[col], errors='coerce')
            if numeric.notna().sum() > 0 and numeric.dtype in [np.float64, np.int64, float, int]:
                candidate_cols.append(col)
        except Exception:
            continue
    if candidate_cols:
        numeric_field_id = candidate_cols[0]  # Use the first numeric field found

if numeric_field_id is None:
    warnings.warn("No suitable numeric field found for EDA. Please check the dataset columns.")
else:
    print(f"Using numeric field '@id': {numeric_field_id}\n")

    # Filter and normalize
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Remove missing/NaN
        filtered_df = df[df[numeric_field_id] > df[numeric_field_id].quantile(0.1)]
        print(f"Filtered DataFrame with '{numeric_field_id}' > 10th percentile:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalization (Z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a categorical grouping field (by unique string count < 25)
        group_field_id = None
        for col in df.columns:
            if col == numeric_field_id:
                continue
            if df[col].dtype == object and df[col].nunique() > 1 and df[col].nunique() <= 25:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped data by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("\nNo suitable grouping field found.")
    except Exception as e:
        print(f"Error during EDA: {e}")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if df is not None and numeric_field_id is not None:
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='teal')
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If a group_field_id was found, show boxplot
    try:
        if 'group_field_id' in locals() and group_field_id:
            plt.figure(figsize=(10, 5))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, palette='Set2')
            plt.title(f'{numeric_field_id} by {group_field_id}')
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f"Boxplot visualization error: {e}")
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

This notebook demonstrated loading and exploring a Croissant-described dataset using `mlcroissant`. We summarized the dataset, listed available data, performed basic exploratory data analysis, and visualized field distributions.

Key findings and EDA results are dataset-dependent. For detailed research, further data cleaning and domain-specific analysis is recommended.